# Empirical Validation of Theory

---

## Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

from networks.BP_network import BP_network
from networks.EWC_network import EWC_network
from networks.EFC_network import EFC_network
from networks.EHC_network import EHC_network
from src.dataloaders import ClassILMNIST2Task
from src.utils import dotdict

## Config

In [17]:
config = {
    "lr": 1e-4,
    "batch_size": 256,
    "epochs": 10,
    "mode": "di",  # or "di"
    "num_workers": 8,
    "loss_fn": "ce", # "mse"
    "optimizer": "Adam",
    "scheduler": "CosineAnnealingLR",
    "device": "cpu",
    "output_dir": "./outputs",
    "seed": 0,
    "target_lr": 1e-2, # needs to be < time_constant_ratio
    "alpha_di": 0.0017,
    "alpha_I": 0.0017,
    "tau": 0.032,
    "dt_di": 0.02,
    "psi_lr": 0.1,
    "alpha_psi": 0.0,
    "time_constant_ratio": 0.2, # this param can be merged with dt_di
    "tmax_di": 500,
    "k_p": 2.0,
    "eps": 1e-4, # there is an interplay between dt_di and eps and between target_lr and eps
    "save": False,
    "importance_ewc": 4.0, # ewc params
    "beta_efc": 0.1, # efc params
    "flatten_imgs": True,
    "setting": "classIL2task", # domainIL, taskIL, classIL5task, classIL2task
    "peak": False, # saves the peak model based on cumulative accuracy and restores it after each task
    "layers": [784, 64, 64, 10],
    "num_tasks": 2,
    "classes_per_task": 5,
}

config = dotdict(config)

torch.manual_seed(config.seed)
np.random.seed(config.seed)

## Dataloaders – Class-IL 2-task Split-MNIST

In [18]:
dataloader = ClassILMNIST2Task(config)

train_loader_A, test_loader_A = dataloader.get_dataloaders(task_id=0)
train_loader_B, test_loader_combined = dataloader.get_dataloaders(task_id=1)

def split_loader(loader, keep_classes):
    """Return a DataLoader that contains only the given class indices."""
    data, targets = [], []
    for x, y in loader:
        idx = (y.argmax(1) >= keep_classes[0]) & (y.argmax(1) <= keep_classes[1])
        data.append(x[idx])
        targets.append(y[idx])
    if not data: return None
    dataset = TensorDataset(torch.cat(data), torch.cat(targets))
    return DataLoader(dataset, batch_size=config.batch_size,
                      shuffle=False, num_workers=config.num_workers, pin_memory=True)

test_loader_B = split_loader(test_loader_combined, [5,9])

print(f"Task A: {len(train_loader_A.dataset)} train, {len(test_loader_A.dataset)} test")
print(f"Task B: {len(train_loader_B.dataset)} train, {len(test_loader_B.dataset)} test_only")
print(f"Combined test: {len(test_loader_combined.dataset)}")

Task A: 30596 train, 5139 test
Task B: 29404 train, 4861 test_only
Combined test: 10000


## Train Task A on BP_network

In [19]:
# Initialize
bp_net_A = BP_network(config).to(config.device)
optimizer = optim.Adam(bp_net_A.parameters(), lr=config.lr)
criterion = nn.CrossEntropyLoss()

def train_one_epoch(net, loader, opt, crit):
    net.train()
    total_loss = 0
    correct = 0
    for x, y in loader:
        x, y = x.to(config.device), y.to(config.device)
        y = y[:, :5]  # only first 5 classes
        opt.zero_grad()
        net.task_id = 0
        out = net(x)
        loss = crit(out, y.argmax(dim=1))
        loss.backward()
        opt.step()
        total_loss += loss.item()
        correct += (out.argmax(1) == y.argmax(1)).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

# Train Task A
epochs = 20
print("Training Task A (BP)...")
for epoch in range(epochs):
    loss, acc = train_one_epoch(bp_net_A, train_loader_A, optimizer, criterion)
    print(f"Epoch {epoch+1:02d} | Loss: {loss:.4f} | Acc: {acc:.4f}")

Training Task A (BP)...
Epoch 01 | Loss: 0.9527 | Acc: 0.6733
Epoch 02 | Loss: 0.2386 | Acc: 0.9336
Epoch 03 | Loss: 0.1574 | Acc: 0.9548
Epoch 04 | Loss: 0.1262 | Acc: 0.9636
Epoch 05 | Loss: 0.1087 | Acc: 0.9685
Epoch 06 | Loss: 0.0962 | Acc: 0.9720
Epoch 07 | Loss: 0.0872 | Acc: 0.9748
Epoch 08 | Loss: 0.0795 | Acc: 0.9769
Epoch 09 | Loss: 0.0733 | Acc: 0.9788
Epoch 10 | Loss: 0.0680 | Acc: 0.9806
Epoch 11 | Loss: 0.0633 | Acc: 0.9817
Epoch 12 | Loss: 0.0589 | Acc: 0.9836
Epoch 13 | Loss: 0.0552 | Acc: 0.9848
Epoch 14 | Loss: 0.0517 | Acc: 0.9857
Epoch 15 | Loss: 0.0485 | Acc: 0.9867
Epoch 16 | Loss: 0.0456 | Acc: 0.9875
Epoch 17 | Loss: 0.0431 | Acc: 0.9882
Epoch 18 | Loss: 0.0405 | Acc: 0.9891
Epoch 19 | Loss: 0.0379 | Acc: 0.9895
Epoch 20 | Loss: 0.0357 | Acc: 0.9901


## Copy Weights to All Methods & Reset Optimizers

In [ ]:
def copy_weights(src_net, dst_net):
    dst_net.load_state_dict(deepcopy(src_net.state_dict()))

# Create all networks
ewc_net = EWC_network(config).to(config.device)
efc_net = EFC_network(config).to(config.device)
bp_net = BP_network(config).to(config.device)
ehc_net = EHC_network(config).to(config.device)

# Copy Task A weights
copy_weights(bp_net_A, ewc_net)
copy_weights(bp_net_A, efc_net)
copy_weights(bp_net_A, bp_net)
copy_weights(bp_net_A, ehc_net)

# Reset optimizers
optimizers = {
    'ewc': optim.Adam(ewc_net.parameters(), lr=config.lr),
    'efc': optim.Adam(efc_net.parameters(), lr=config.lr),
    'bp':  optim.Adam(bp_net.parameters(), lr=config.lr),
    'ehc': optim.Adam(ehc_net.parameters(), lr=config.lr),
}

print("Weights copied, optimizers reset.")

Weights copied, optimizers reset.


## Compute Fisher & Hessian

In [ ]:
ewc_net.task_id = 0
efc_net.task_id = 0
bp_net.task_id = 0
ehc_net.task_id = 0

ewc_net.complete_task(train_loader_A)
efc_net.complete_task(train_loader_A)
bp_net.complete_task(train_loader_A)
ehc_net.complete_task(train_loader_A)

ewc_net.task_id = 1
efc_net.task_id = 1
bp_net.task_id = 1
ehc_net.task_id = 1

fisher_A = ewc_net._fisher
hessian_A = ehc_net._hessian

## Train and evaluate on Task B

In [ ]:
def evaluate_acc(net, loader, task_id, offset=0):
    """offset = 5 for Task-B-only (maps predictions 0-4 → 5-9)."""
    net.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(config.device), y.to(config.device)
            net.task_id = task_id
            out = net(x)
            pred = out.argmax(1) + offset
            correct += (pred == y.argmax(1)).sum().item()
            total   += y.size(0)
    return correct / total if total else 0.0

In [ ]:
nets = {
    'BP': bp_net,
    'EWC': ewc_net,
    'EFC': efc_net,
    'EHC': ehc_net
}

weight_deltas = {name: [] for name in nets}  # List of dicts {param_name: delta_tensor} for first 5 batches
acc_logs = {name: {'acc_A': [], 'acc_B': [], 'combined': []} for name in nets}

def train_on_B(epochs=10):
    for name, net in nets.items():
        print(f"Training {name} on Task B...")
        opt = optimizers[name]
        net.train()
        batch_idx = 0
        for _ in range(epochs):
            for x, y in train_loader_B:
                x, y = x.to(config.device), y.to(config.device)
                net.task_id = 1
                old_params = {n: p.clone().detach() for n, p in net.named_parameters() if p.requires_grad}
                opt.zero_grad()
                _ = net(x)
                net.backward(y)
                opt.step()
                # Track delta for first 5 batches
                if batch_idx < 5:
                    delta = {n: p - old_params[n] for n, p in net.named_parameters() if p.requires_grad}
                    weight_deltas[name].append(delta)
                # Log accs per batch
                acc_A = evaluate_acc(net, test_loader_A, task_id=0, offset=0)
                acc_B = evaluate_acc(net, test_loader_B, task_id=1, offset=5)
                combined = evaluate_acc(net, test_loader_combined, task_id=1, offset=0)
                acc_logs[name]['acc_A'].append(acc_A)
                acc_logs[name]['acc_B'].append(acc_B)
                acc_logs[name]['combined'].append(combined)
                batch_idx += 1

train_on_B(epochs=20)